# 🎙️ Shiro Voice Changer Real-Time (RVC MMVC Server)
Notebook resmi untuk menjalankan **Real-Time Voice Changer (RVC)** Shiro di Google Colab dengan akselerasi GPU (T4 / A100).

### 🌟 Dua Colab Resmi Shiro LLMA Project:
1. 🧠 [**Shiro_LLMA_Colab.ipynb**](https://colab.research.google.com/github/Renn-devBI/ShiroLLM-AI-CHAT/blob/main/Shiro_LLMA_Colab.ipynb) — Otak Utama AI Percakapan, Multimodal Vision, & API Gateway.
2. 🎙️ [**Shiro_VoiceChanger_Colab.ipynb**](https://colab.research.google.com/github/Renn-devBI/ShiroLLM-AI-CHAT/blob/main/Shiro_VoiceChanger_Colab.ipynb) — Voice Changer Real-Time RVC (Notebook ini).

> [!TIP]
> Anda dapat menjalankan kedua notebook ini secara bersamaan di 2 tab Colab terpisah untuk mendapatkan pengalaman VTuber AI penuh: Shiro AI merespons teks/suara dan suara Anda diubah menjadi suara karakter secara real-time!

## Langkah 1: Cek & Verifikasi GPU
Jalankan cell ini untuk memastikan Colab terhubung ke GPU (T4 atau A100).

In [ ]:
import subprocess
res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if res.returncode == 0:
    print("=" * 60)
    print("GPU AKTIF: Siap untuk Voice Changer Real-Time!")
    print("=" * 60)
    print(res.stdout.split('\n')[0])
    for line in res.stdout.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'A100' in line or 'V100' in line or 'NVIDIA' in line:
            print(line)
    print("=" * 60)
else:
    print("=" * 60)
    print("PERINGATAN: GPU BELUM DIAKTIFKAN!")
    print("Voice Changer membutuhkan GPU untuk inferensi real-time rendah latency.")
    print("Buka menu Runtime -> Change runtime type -> Pilih T4 GPU -> Simpan.")
    print("=" * 60)

## Langkah 2: Unduh Voice Changer & Pasang Dependensi
Tahap ini akan mengunduh versi terbaru w-okada Voice Changer dan menginstall dependensi audio (FAISS GPU CUDA 12, PyWorld, Fairseq, ONNXRuntime GPU).

Proses instalasi membutuhkan waktu sekitar **2-3 menit**.

In [ ]:
# ==============================================================================
# @title **[1]** Setup Virtual Environment (VENV) & Pasang Dependensi
# @markdown Membuat venv Python 3.10 mandiri (`/content/vc_venv`) tanpa mengubah Python bawaan Colab. **Proses ±5-10 menit.**
# ==============================================================================
import os
import sys
import shutil
import codecs

# 1. Deteksi lingkungan
notebook_env = 1 if os.path.exists('/content') else (2 if os.path.exists('/kaggle/working') else 3)
BASE_DIR = '/content' if notebook_env == 1 else os.getcwd()
os.chdir(BASE_DIR)
print(f"> Lingkungan Terdeteksi: {'Google Colab' if notebook_env == 1 else ('Kaggle' if notebook_env == 2 else 'Local/Other')}")

# @markdown ---
# @markdown **[Opsional]** Simpan model di Google Drive agar tidak hilang saat runtime terputus.
Use_Drive = False # @param {type:"boolean"}

DRIVE_MODEL_DIR = "/content/drive/MyDrive/Shiro_VoiceChanger/model_dir"
if notebook_env == 1 and Use_Drive:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    print(f"> [Drive] Penyimpanan Model Permanen: {DRIVE_MODEL_DIR}")

# 2. Paket sistem audio
print("> [1/7] Memasang paket sistem audio (portaudio, libsndfile, ffmpeg)...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq build-essential libportaudio2 portaudio19-dev libsndfile1 psmisc ffmpeg curl > /dev/null 2>&1
!npm install -g @hrzn/cli > /dev/null 2>&1 || true

# 3. Tools ringan untuk kernel notebook (pakai interpreter kernel itu sendiri, bukan /usr/local/bin/pip)
print("> [2/7] Memasang helper kernel (pyngrok, portpicker)...")
HOST_PY = sys.executable
!{HOST_PY} -m pip install -q pyngrok portpicker colorama

# 4. Astral uv (hanya dipakai untuk membuat venv Python 3.10)
print("> [3/7] Memasang Astral uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:/root/.cargo/bin:{os.environ.get('PATH', '')}"
# Paksa compiler sistem untuk build ekstensi C (pyworld, fairseq) di Python standalone uv
os.environ['CC'] = 'gcc'
os.environ['CXX'] = 'g++'
UV_CMD = "/root/.local/bin/uv" if os.path.exists("/root/.local/bin/uv") else "uv"

# 5. Buat venv Python 3.10
VENV_DIR = f"{BASE_DIR}/vc_venv"
VENV_PY = f"{VENV_DIR}/bin/python"
print(f"> [4/7] Membuat venv Python 3.10 ({VENV_DIR})...")
if not os.path.exists(VENV_PY):
    !{UV_CMD} venv {VENV_DIR} --python 3.10 --seed -q
if not os.path.exists(VENV_PY):
    !apt-get install -y -qq python3.10 python3.10-venv python3.10-dev > /dev/null 2>&1
    !python3.10 -m venv {VENV_DIR}
if not os.path.exists(VENV_PY):
    raise RuntimeError("Gagal membuat venv Python 3.10. Coba Runtime > Disconnect and delete runtime, lalu jalankan ulang.")
!{VENV_PY} --version

# pip < 24.1 WAJIB: fairseq -> omegaconf 2.0.x punya metadata lama yang ditolak pip >= 24.1
!{VENV_PY} -m ensurepip -q > /dev/null 2>&1 || true
!{VENV_PY} -m pip install -q "pip==23.3.1" "setuptools<70" wheel "Cython>=3.0" "numpy<2.0.0,>=1.24.0"

# 6. PyTorch CUDA 12.1
print("> [5/7] Memasang PyTorch GPU CUDA 12.1 ke dalam venv...")
!{VENV_PY} -m pip install -q torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# 7. Clone Voice Changer
print("> [6/7] Mengunduh repository Voice Changer...")
externalgit = codecs.decode('uggcf://tvguho.pbz/j-bxnqn/ibvpr-punatre.tvg', 'rot_13')
pathloc = codecs.decode('ibvpr-punatre', 'rot_13')
os.chdir(BASE_DIR)
if not os.path.exists(pathloc):
    !git clone --depth 1 {externalgit} > /dev/null 2>&1
else:
    print(f"  Folder {pathloc} sudah ada, dilewati.")
os.chdir(f"{BASE_DIR}/{pathloc}/server")

version_file = '../client/demo/dist/assets/gui_settings/version.txt'
if os.path.exists(version_file):
    !sed -i "s/-.-.-.-/Shiro.Voice.Mod/" {version_file}

# 8. Dependensi inti & audio (semua di dalam venv)
print("> [7/7] Memasang Fairseq, PyWorld, FAISS, ONNXRuntime & pustaka audio ke venv...")
!{VENV_PY} -m pip install -q "faiss-gpu-cu12<1.10" || {VENV_PY} -m pip install -q "faiss-cpu<1.10"
!{VENV_PY} -m pip install -q onnxruntime-gpu

# PyWorld tidak punya wheel Linux -> harus di-compile (butuh gcc + header Python).
# Urutan: build terisolasi versi terbaru -> build tanpa isolasi (Cython 3) -> versi lama.
!{VENV_PY} -m pip install -q pyworld \
  || {VENV_PY} -m pip install -q "pyworld==0.3.4" --no-build-isolation \
  || {VENV_PY} -m pip install -q "pyworld==0.3.2" --no-build-isolation

!{VENV_PY} -m pip install -q fairseq==0.12.2
!{VENV_PY} -m pip install -q sounddevice soundfile librosa torchcrepe torchfcpe noisereduce pedalboard gdown tensorboardX

# requirements.txt bawaan: buang paket yang sudah kita pasang versi GPU-nya
!sed -i -E '/^(torch|torchaudio|numpy|onnxruntime|onnxruntime-gpu|faiss-cpu|faiss-gpu|fairseq|pyworld|setuptools)([=<>~ ]|$)/d' requirements.txt
!sed -i '/^mat$/d' requirements.txt
!{VENV_PY} -m pip install -q -r requirements.txt

# Kunci ulang versi kritis (paket lain sering menaikkannya):
#  - numpy < 2     : ABI fairseq / pyworld / faiss
#  - setuptools<70 : librosa lama butuh `pkg_resources` (dihapus di setuptools baru)
!{VENV_PY} -m pip install -q "numpy<2.0.0,>=1.24.0" "setuptools<70"

# Pastikan pyworld benar-benar ada (pasang ulang jika tertimpa)
!{VENV_PY} -c "import pyworld" 2>/dev/null || {VENV_PY} -m pip install -q --force-reinstall --no-deps pyworld

# 9. Folder model (opsional ke Google Drive)
if notebook_env == 1 and Use_Drive:
    if os.path.exists('model_dir') and not os.path.islink('model_dir'):
        shutil.rmtree('model_dir', ignore_errors=True)
    if not os.path.exists('model_dir'):
        os.symlink(DRIVE_MODEL_DIR, 'model_dir')
else:
    os.makedirs('model_dir', exist_ok=True)

# 10. Verifikasi ketat: cek setiap modul yang dibutuhkan server
print("> Verifikasi modul di dalam venv:")
import subprocess
verify_code = (
    "import importlib, sys\n"
    "mods = ['pkg_resources','numpy','torch','torchaudio','sounddevice','soundfile','librosa',"
    "'pyworld','fairseq','faiss','onnxruntime','torchcrepe','socketio','uvicorn','fastapi']\n"
    "bad = []\n"
    "for m in mods:\n"
    "    try:\n"
    "        importlib.import_module(m); print('  [OK]  ', m)\n"
    "    except Exception as e:\n"
    "        bad.append(m); print('  [GAGAL]', m, '->', type(e).__name__, e)\n"
    "import torch, numpy\n"
    "print('  torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| numpy', numpy.__version__)\n"
    "sys.exit(1 if bad else 0)\n"
)
res = subprocess.run([VENV_PY, "-W", "ignore", "-c", verify_code], cwd=os.getcwd(), capture_output=True, text=True)
print("\n".join(l for l in res.stdout.splitlines() if l.startswith("  ")))

if res.returncode == 0:
    Run_Cell = 1
    print("\033[92m=================================================================\033[0m")
    print("\033[92m> [SUKSES] Virtual Environment & Voice Changer SIAP!\033[0m")
    print("\033[92m=================================================================\033[0m")
    print("\033[96m> Lanjut ke Langkah 2 (model suara, opsional) atau Langkah 3 (jalankan server).\033[0m")
else:
    Run_Cell = 0
    print("\033[91m> [GAGAL] Ada modul yang belum terpasang (lihat [GAGAL] di atas).\033[0m")
    print("\033[91m  Jalankan ulang cell ini. Jika tetap gagal: Runtime > Disconnect and delete runtime.\033[0m")

## Langkah 3: Unggah / Impor Model Suara RVC (Opsional)
Gunakan cell ini jika Anda ingin mengimpor model suara karakter RVC (file `.zip` atau `.pth` dari HuggingFace atau Google Drive).

> [!NOTE]
> Anda juga bisa mengunggah model langsung lewat antarmuka Web UI Voice Changer setelah server berjalan pada Langkah 4.

In [ ]:
# ==============================================================================
# @title **[2] [Opsional]** Upload / Impor Model Suara RVC ke Slot Tertentu
# ==============================================================================
import os
import sys
import zipfile
import shutil

SERVER_DIR = '/content/voice-changer/server'
if os.path.exists(SERVER_DIR):
    os.chdir(SERVER_DIR)
VENV_PY = "/content/vc_venv/bin/python" if os.path.exists("/content/vc_venv/bin/python") else sys.executable

model_slot = 1 # @param {type:"number"}
model_slot = str(int(model_slot))

# @markdown **[Opsional]** Link gambar icon model (URL gambar png/jpg):
icon_link = "https://i.scdn.co/image/ab6761610000e5ebbab6de14398281b2d37bc0b5" # @param {type:"string"}

# @markdown **Link Model RVC (.zip):** HuggingFace (resolve link) atau Google Drive:
model_link = "https://huggingface.co/megaaziib/my-rvc-models-collection/resolve/main/zetaTest.zip?download=true" # @param {type:"string"}

# @markdown **Pengaturan Pitch & Indeks Suara:**
Tune = 12 # @param {type:"slider", min:-50, max:50, step:1}
Index = 0 # @param {type:"slider", min:0, max:1, step:0.1}

slot_dir = os.path.join("model_dir", model_slot)
shutil.rmtree(slot_dir, ignore_errors=True)
os.makedirs(slot_dir, exist_ok=True)

# 1. Unduh model
print(f"> Mengunduh model RVC untuk Slot {model_slot}...")
model_zip_path = "model.zip"
if os.path.exists(model_zip_path):
    os.remove(model_zip_path)

link = model_link.strip()
if "weights.gg" in link:
    print("\033[93m[Info] Link weights.gg tidak didukung. Gunakan HuggingFace atau Google Drive.\033[0m")
elif link.startswith("https://drive.google.com"):
    print("  Sumber: Google Drive")
    !{VENV_PY} -m gdown "{link}" -O "{model_zip_path}" --fuzzy -q
else:
    print("  Sumber: HuggingFace / link langsung")
    !curl -L -s "{link}" -o "{model_zip_path}"

# 2. Ekstrak .pth / .onnx / .index (abaikan struktur folder di dalam zip)
extracted = 0
if os.path.exists(model_zip_path) and os.path.getsize(model_zip_path) > 1000:
    try:
        with zipfile.ZipFile(model_zip_path, 'r') as zf:
            for member in zf.infolist():
                fname = os.path.basename(member.filename)
                if fname.lower().endswith(('.pth', '.onnx', '.index')) and not fname.startswith('._'):
                    with zf.open(member) as src, open(os.path.join(slot_dir, fname), "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    extracted += 1
    except zipfile.BadZipFile:
        # Bukan zip -> anggap file .pth langsung
        shutil.move(model_zip_path, os.path.join(slot_dir, "model.pth"))
        extracted = 1
    if os.path.exists(model_zip_path):
        os.remove(model_zip_path)
    print(f"  Berhasil mengekstrak {extracted} berkas model.")
else:
    print("\033[91m[Error] Unduhan gagal / file kosong. Periksa link model.\033[0m")

# 3. Icon
icon_file_name = ""
clean_icon = icon_link.strip().strip('"').strip("'")
if clean_icon:
    icon_file_name = "icon.png"
    !curl -L -s "{clean_icon}" -o "{slot_dir}/icon.png"

# 4. paramset.py lokal (tanpa pastebin). Import Voice Changer dibungkus try
#    agar params.json tetap dibuat walaupun modul internal gagal diimpor.
paramset_script = '''import argparse, os, json

parser = argparse.ArgumentParser()
parser.add_argument("--model_slot", type=str, required=True)
parser.add_argument("--iconFile", type=str, default="")
parser.add_argument("--Tune", type=str, default="0")
parser.add_argument("--Index", type=str, default="0")
args = parser.parse_args()

slot_dir = os.path.join("model_dir", args.model_slot)
files = os.listdir(slot_dir)
pth_files = sorted([f for f in files if f.lower().endswith((".pth", ".onnx"))])
index_files = sorted([f for f in files if f.lower().endswith(".index")])
# Prioritaskan index "added_*" (index final hasil training RVC)
index_files.sort(key=lambda f: 0 if f.startswith("added") else 1)

model_file = pth_files[0] if pth_files else ""
index_file = index_files[0] if index_files else ""
model_name = os.path.splitext(model_file)[0] if model_file else "Model_Slot_" + args.model_slot

params = {
    "slotIndex": int(args.model_slot),
    "voiceChangerType": "RVC",
    "name": model_name,
    "description": "Model Suara Shiro RVC",
    "credit": "",
    "termsOfUseUrl": "",
    "iconFile": args.iconFile,
    "speakers": {"0": "target"},
    "modelFile": model_file,
    "indexFile": index_file,
    "defaultTune": int(float(args.Tune)),
    "defaultIndexRatio": float(args.Index),
    "defaultProtect": 0.5,
    "isONNX": model_file.lower().endswith(".onnx"),
    "modelType": "pyTorchRVCv2",
    "samplingRate": 40000,
    "f0": True,
    "embChannels": 768,
    "embOutputLayer": 12,
    "useFinalProj": False,
    "deprecated": False,
    "embedder": "hubert_base",
    "sampleId": "",
    "version": "v2",
}

# Baca metadata asli model (.pth) untuk samplingRate / versi / f0
try:
    import torch
    if model_file.lower().endswith(".pth"):
        cpt = torch.load(os.path.join(slot_dir, model_file), map_location="cpu")
        cfg = cpt.get("config", [])
        if cfg:
            params["samplingRate"] = int(cfg[-1])
        version = cpt.get("version", "v1")
        params["f0"] = bool(cpt.get("f0", 1))
        if version == "v2":
            params["version"] = "v2"
            params["embChannels"] = 768
            params["embOutputLayer"] = 12
            params["useFinalProj"] = False
            params["modelType"] = "pyTorchRVCv2" if params["f0"] else "pyTorchRVCv2Nono"
        else:
            params["version"] = "v1"
            params["embChannels"] = 256
            params["embOutputLayer"] = 9
            params["useFinalProj"] = True
            params["modelType"] = "pyTorchRVC" if params["f0"] else "pyTorchRVCNono"
except Exception as e:
    print("  [Info] Metadata model tidak terbaca, pakai default v2/40k:", e)

with open(os.path.join(slot_dir, "params.json"), "w", encoding="utf-8") as f:
    json.dump(params, f, indent=2)

print("> Slot %s: %s | %s | %d Hz | index: %s" % (args.model_slot, model_name, params["version"], params["samplingRate"], index_file or "-"))
'''

with open('paramset.py', 'w', encoding='utf-8') as f:
    f.write(paramset_script)

if extracted > 0:
    !{VENV_PY} paramset.py --model_slot "{model_slot}" --iconFile "{icon_file_name}" --Tune "{Tune}" --Index "{Index}"
    print(f"\033[92m> [SUKSES] Model suara berhasil disiapkan pada Slot {model_slot}!\033[0m")

## Langkah 4: Jalankan Server Voice Changer
Pilih opsi tunneling yang diinginkan:
- **CLOUDFLARE (Rekomendasi ⭐)**: 100% Gratis, tanpa perlu daftar akun/token, dan tidak memakan kuota session Ngrok.
- **NGROK**: Jika Anda memiliki token dari [dashboard.ngrok.com](https://dashboard.ngrok.com).
- **HRZN**: Jika Anda memiliki token dari [hrzn.run](https://hrzn.run).

Saat pertama kali dijalankan, server akan mengunduh model embedding dasar (~1-2 menit). Tunggu hingga link URL publik muncul!

In [ ]:
# ==============================================================================
# @title **[3]** Jalankan Server Voice Changer (CLOUDFLARE / NGROK / HRZN)
# @markdown Pertama kali dijalankan server akan mengunduh pretrain model (±1-2 menit).
# ==============================================================================
import os
import re
import sys
import time
import socket
import shutil
import codecs
import threading
import subprocess
import portpicker
from IPython.display import clear_output, display, Audio

SERVER_DIR = '/content/voice-changer/server'
if os.path.exists(SERVER_DIR):
    os.chdir(SERVER_DIR)
VENV_PY = "/content/vc_venv/bin/python" if os.path.exists("/content/vc_venv/bin/python") else sys.executable

# Hapus sisa runner versi lama (penyebab SyntaxError 'unterminated string literal')
if os.path.exists("run_server.py"):
    os.remove("run_server.py")

# Self-heal: pastikan modul kritis ada sebelum server dijalankan
!{VENV_PY} -W ignore -c "import pkg_resources" 2>/dev/null || {VENV_PY} -m pip install -q "setuptools<70"
!{VENV_PY} -W ignore -c "import pyworld" 2>/dev/null || {VENV_PY} -m pip install -q pyworld

# ====================== Konfigurasi Tunnel ======================
TUNNEL = "CLOUDFLARE" # @param ["CLOUDFLARE", "NGROK", "HRZN"]
# @markdown Token hanya untuk NGROK / HRZN. Kosongkan untuk memakai Colab Secret `NGROK_TOKEN` atau input manual.
Token = "" # @param {type:"string"}
Region = "ap - Asia/Pacific (Singapore)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)", "eu - Europe (Frankfurt)", "in - India (Mumbai)", "jp - Japan (Tokyo)", "sa - South America (Sao Paulo)", "us - United States (Ohio)"]
ClearConsole = True # @param {type:"boolean"}
Play_Notification = False # @param {type:"boolean"}

if TUNNEL in ("NGROK", "HRZN") and not Token.strip():
    try:
        from google.colab import userdata
        Token = userdata.get('NGROK_TOKEN' if TUNNEL == "NGROK" else 'HRZN_TOKEN') or ""
    except Exception:
        Token = ""
    if not Token.strip():
        Token = input(f"Masukkan {TUNNEL} Authtoken: ").strip()

PORT = portpicker.pick_unused_port()
public_url = ""
ngrok = None

# Regex tanpa backslash agar bebas SyntaxWarning
CF_PATTERN = r"https://[-a-zA-Z0-9.]+[.]trycloudflare[.]com"
HRZN_PATTERN = r"https://[-a-zA-Z0-9.]+[.]hrzn[.]run"

# 1. Tunnel publik
if TUNNEL == "CLOUDFLARE":
    print("> Mengaktifkan Cloudflare Tunnel (gratis, tanpa token)...")
    if not os.path.exists('/content/cloudflared'):
        !curl -s -L -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /content/cloudflared
    !pkill -f cloudflared > /dev/null 2>&1 || true
    cf_log = '/content/cf_tunnel.log'
    cf_log_fh = open(cf_log, 'w')
    subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
                     stdout=cf_log_fh, stderr=subprocess.STDOUT)
    start_t = time.time()
    while time.time() - start_t < 40 and not public_url:
        time.sleep(1)
        with open(cf_log, 'r', encoding='utf-8', errors='ignore') as f:
            m = re.findall(CF_PATTERN, f.read())
        if m:
            public_url = m[0]

elif TUNNEL == "NGROK":
    try:
        from pyngrok import conf, ngrok
    except ImportError:
        !{sys.executable} -m pip install -q pyngrok
        from pyngrok import conf, ngrok
    MyConfig = conf.PyngrokConfig()
    MyConfig.auth_token = Token.strip()
    MyConfig.region = Region[0:2]
    conf.set_default(MyConfig)
    try:
        ngrok.kill()
    except Exception:
        pass
    public_url = ngrok.connect(PORT).public_url

elif TUNNEL == "HRZN":
    if os.path.exists('url.txt'):
        os.remove('url.txt')
    !hrzn login {Token.strip()}
    url_fh = open('url.txt', 'w')
    subprocess.Popen(['hrzn', 'tunnel', f'http://localhost:{PORT}'], stdout=url_fh, stderr=subprocess.STDOUT)
    start_t = time.time()
    while time.time() - start_t < 20 and not public_url:
        time.sleep(1)
        with open('url.txt', 'r', errors='ignore') as f:
            m = re.findall(HRZN_PATTERN, f.read())
        if m:
            public_url = m[0]

if not public_url:
    print("\033[91m[Error] Gagal mendapatkan URL publik dari tunnel. Coba jalankan ulang cell ini atau ganti TUNNEL.\033[0m")
else:
    print(f"> URL publik: {public_url}")

    # 2. Rename MMVCServerSIO.py -> HVoice.py (aman dijalankan berulang)
    mainpy = codecs.decode('ZZIPFreireFVB.cl', 'rot_13')  # MMVCServerSIO.py
    if os.path.exists(mainpy):
        if os.path.exists("HVoice.py"):
            os.remove("HVoice.py")
        shutil.move(mainpy, "HVoice.py")
        !sed -i "s/MMVCServerSIO/HVoice/g" HVoice.py
    os.makedirs("pretrain", exist_ok=True)
    os.makedirs("model_dir", exist_ok=True)

    # Unduh pretrain models penting jika belum ada agar pipeline langsung instan & tidak looping "Waiting generate pipeline"
    print("> Memeriksa model pretrain (RMVPE & HuBERT)...")
    if not os.path.exists("pretrain/rmvpe.onnx") or os.path.getsize("pretrain/rmvpe.onnx") < 1000000:
        !curl -L -s -o pretrain/rmvpe.onnx "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.onnx"
    if not os.path.exists("pretrain/rmvpe.pt") or os.path.getsize("pretrain/rmvpe.pt") < 1000000:
        !curl -L -s -o pretrain/rmvpe.pt "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt"
    if not os.path.exists("pretrain/hubert_base.pt") or os.path.getsize("pretrain/hubert_base.pt") < 1000000:
        !curl -L -s -o pretrain/hubert_base.pt "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt"

    # 3. Notifikasi saat server siap
    def wait_for_server():
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', PORT))
            sock.close()
            if result == 0:
                break
        if ClearConsole:
            clear_output()
        print("=" * 64)
        print("  \033[92m\033[1mSERVER VOICE CHANGER BERHASIL BERJALAN!\033[0m")
        print("=" * 64)
        print("  Akses Web UI Voice Changer:")
        print(f"  -> \033[94m\033[1m{public_url}\033[0m")
        print("=" * 64)
        print("  1. Buka link di atas di browser Anda.")
        print("  2. Pilih microphone & speaker di Web UI.")
        print("  3. Tekan START untuk mulai mengubah suara secara real-time!")
        print("  Saran: f0 RMVPE_ONNX | Chunk 112+ | Extra 8192")
        print("=" * 64)
        if Play_Notification:
            display(Audio(url='https://raw.githubusercontent.com/hinabl/rmvpe-ai-kaggle/main/custom/audios/notif.mp3', autoplay=True))

    threading.Thread(target=wait_for_server, daemon=True).start()

    # 4. Jalankan server dengan Python VENV
    !{VENV_PY} HVoice.py \
        -p {PORT} \
        --https False \
        --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
        --content_vec_500_onnx pretrain/content_vec_500.onnx \
        --content_vec_500_onnx_on false \
        --hubert_base pretrain/hubert_base.pt \
        --hubert_base_jp pretrain/rinna_hubert_base_jp.pt \
        --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
        --nsf_hifigan pretrain/nsf_hifigan/model \
        --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
        --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
        --rmvpe pretrain/rmvpe.pt \
        --model_dir model_dir \
        --samples samples.json \
        --allowed-origins "{public_url}"

# 5. Cleanup
if TUNNEL == "NGROK" and ngrok is not None:
    try:
        ngrok.kill()
    except Exception:
        pass
elif TUNNEL == "CLOUDFLARE":
    !pkill -f cloudflared > /dev/null 2>&1 || true
elif TUNNEL == "HRZN":
    !fuser -k {PORT}/tcp > /dev/null 2>&1 || true
print("\n--------- SERVER VOICE CHANGER SELESAI ---------")